# Evaluation: Base Qwen2.5-Coder vs Fine-tuned V1

This notebook compares the original `Qwen/Qwen2.5-Coder-1.5B-Instruct` model with the local V1 merged model trained for 300 steps. The purpose is to test whether fine-tuning produced practical value for short Python function generation.

**Comparison scope:** deterministic generation, identical prompts, identical decoding settings, one model loaded at a time to fit the available VRAM.


## 1. Imports and Configuration

This cell defines the base model, the local V1 model path, and the ten evaluation prompts. The prompts focus on common Python function tasks so both models can be compared under the same conditions.


In [1]:
import torch
import json
import time
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path

BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
V1_PATH = Path("./mini-gpt-coder-merged")

PROMPTS = [
    "Write a Python function to add two numbers",
    "Write a Python function to reverse a string",
    "Write a Python function to print numbers from 1 to 10 using a for loop",
    "Write a Python function to find the factorial of a number",
    "Write a Python function to check if a string is a palindrome",
    "Write a Python function to find the maximum element in a list",
    "Write a Python function to count the occurrences of a word in a string",
    "Write a Python function to flatten a nested list",
    "Write a Python function to merge two sorted lists",
    "Write a Python function to check if a number is prime",
]

## 2. Helper Functions

The helper layer standardizes prompt formatting, generation, model loading, memory cleanup, and basic code-quality metrics. The metrics are intentionally lightweight: they are useful for comparing output shape and syntax, but they do not replace human review or unit tests.


In [ ]:
def format_prompt(user_prompt):
    return (
        "<|im_start|>user\n"
        "Write a Python function for the following:\n"
        f"{user_prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

def clean_output(decoded, prompt):
    output = decoded.replace(prompt, "")
    output = output.split("<|im_end|>")[0]
    output = output.replace("<|im_start|>assistant", "")
    return output.strip()

def generate(model, tokenizer, prompt_text, max_new_tokens=256):
    prompt = format_prompt(prompt_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    stop_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    start = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=stop_token_id,
        )
    elapsed = time.time() - start

    decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
    output = clean_output(decoded, prompt)
    tokens_generated = generated_ids.shape[1] - inputs["input_ids"].shape[1]
    return output, elapsed, tokens_generated

def load_local_model(path):
    tokenizer = AutoTokenizer.from_pretrained(path)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        path,
        dtype=torch.float16,
        device_map="auto",
        max_memory={0: "4GiB", "cpu": "16GiB"},
        low_cpu_mem_usage=True,
    )
    model.eval()
    return model, tokenizer

def load_base_model():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        dtype=torch.float16,
        device_map="auto",
        max_memory={0: "4GiB", "cpu": "16GiB"},
        low_cpu_mem_usage=True,
    )
    model.eval()
    return model, tokenizer

def unload_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

def compute_metrics(code):
    """Compute basic code quality metrics from generated output."""
    lines = code.strip().splitlines()
    has_docstring = '"""' in code or "'''" in code
    has_type_hints = "->" in code or ": int" in code or ": str" in code or ": list" in code or ": float" in code
    has_edge_case = "if" in code and any(x in code for x in ["None", "0", "[]", '""', "< 0", "== 0", "len("])
    has_return = "return" in code
    num_lines = len([l for l in lines if l.strip()])
    is_syntactically_valid = True
    try:
        compile(code, "<string>", "exec")
    except SyntaxError:
        is_syntactically_valid = False
    return {
        "has_docstring": has_docstring,
        "has_type_hints": has_type_hints,
        "has_edge_case_handling": has_edge_case,
        "has_return_statement": has_return,
        "lines_of_code": num_lines,
        "syntactically_valid": is_syntactically_valid,
    }

: 

## 3. Base Model Evaluation

This section runs the unmodified Qwen2.5-Coder model against every prompt. It records generated text, latency, token count, throughput, and basic quality metrics for later comparison.


In [ ]:
print("=" * 60)
print("EVALUATING: Base Model (Qwen2.5-Coder-1.5B-Instruct)")
print("=" * 60)

model, tokenizer = load_base_model()
base_results = {}

for prompt in PROMPTS:
    print(f"\nPrompt: {prompt}")
    output, elapsed, tokens = generate(model, tokenizer, prompt)
    metrics = compute_metrics(output)
    base_results[prompt] = {
        "output": output,
        "time_seconds": round(elapsed, 2),
        "tokens_generated": tokens,
        "tokens_per_second": round(tokens / elapsed, 2),
        "metrics": metrics,
    }
    print(output)
    print(f"Time: {elapsed:.2f}s | Tokens: {tokens} | tok/s: {tokens/elapsed:.1f}")
    print(f"Metrics: {metrics}")
    print("-" * 40)

unload_model(model)
print("\nBase model unloaded.")

EVALUATING: Base Model (Qwen2.5-Coder-1.5B-Instruct)


## 4. Fine-tuned V1 Evaluation

This section evaluates the local `mini-gpt-coder-merged` model using the same prompts and generation settings. The base model is unloaded before this step so the notebook can run on limited VRAM.


In [ ]:
print("=" * 60)
print("EVALUATING: V1 Fine-tuned (300 steps, 8% of dataset)")
print("=" * 60)

model, tokenizer = load_local_model(V1_PATH)
v1_results = {}

for prompt in PROMPTS:
    print(f"\nPrompt: {prompt}")
    output, elapsed, tokens = generate(model, tokenizer, prompt)
    metrics = compute_metrics(output)
    v1_results[prompt] = {
        "output": output,
        "time_seconds": round(elapsed, 2),
        "tokens_generated": tokens,
        "tokens_per_second": round(tokens / elapsed, 2),
        "metrics": metrics,
    }
    print(output)
    print(f"Time: {elapsed:.2f}s | Tokens: {tokens} | tok/s: {tokens/elapsed:.1f}")
    print(f"Metrics: {metrics}")
    print("-" * 40)

unload_model(model)
print("\nV1 unloaded.")

EVALUATING: V1 Fine-tuned (300 steps, 8% of dataset)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Prompt: Write a Python function to add two numbers
def add(self, x, y):
        """
        Add two numbers.
        
        :param x: First number
        :type x: int or float
        
        :param y: Second number
        :type y: int or float

        :return: Sum of x and y
        :rtype: int or float
        """

        return x + y
Time: 39.20s | Tokens: 76 | tok/s: 1.9
Metrics: {'has_docstring': True, 'has_type_hints': True, 'has_edge_case_handling': False, 'has_return_statement': True, 'lines_of_code': 11, 'syntactically_valid': True}
----------------------------------------

Prompt: Write a Python function to reverse a string
def reverse(s):
    """Reverse a string."""
    return s[::-1]
Time: 8.53s | Tokens: 17 | tok/s: 2.0
Metrics: {'has_docstring': True, 'has_type_hints': False, 'has_edge_case_handling': False, 'has_return_statement': True, 'lines_of_code': 3, 'syntactically_valid': True}
----------------------------------------

Prompt: Write a Python function to pr

## 5. Side-by-side Output Review

This section prints both outputs prompt by prompt. Use it for qualitative review: concision, correctness, formatting style, missing imports, unwanted explanations, and whether the generated code matches the request.


In [ ]:
print("=" * 60)
print("SIDE BY SIDE COMPARISON")
print("=" * 60)

for prompt in PROMPTS:
    print(f"\n{'=' * 60}")
    print(f"PROMPT: {prompt}")
    print(f"\n--- BASE MODEL ---")
    print(base_results[prompt]["output"])
    print(f"\n--- V1 FINE-TUNED ---")
    print(v1_results[prompt]["output"])

SIDE BY SIDE COMPARISON

PROMPT: Write a Python function to add two numbers

--- BASE MODEL ---
Certainly! Below is a simple Python function that takes two parameters and returns their sum:

```python
def add_numbers(num1, num2):
    return num1 + num2

# Example usage:
result = add_numbers(3, 5)
print(result)  # Output: 8
```

### Explanation:
- The function `add_numbers` is defined with two parameters: `num1` and `num2`.
- Inside the function, the addition of `num1` and `num2` is performed using the `+` operator.
- The result of the addition is then returned by the function.

You can call this function with any two numbers you want to add. For example, `add_numbers(3, 5)` will return `8`.

--- V1 FINE-TUNED ---
def add(self, x, y):
        """
        Add two numbers.
        
        :param x: First number
        :type x: int or float
        
        :param y: Second number
        :type y: int or float

        :return: Sum of x and y
        :rtype: int or float
        """

   

## 6. Aggregate Metrics Summary

This summary compares the two result sets across syntax validity, return statements, docstrings, type-hint signals, edge-case handling, average output length, and generation speed. Treat these as directional indicators, not a final benchmark.


In [ ]:
print("=" * 60)
print("AGGREGATE METRICS SUMMARY")
print("=" * 60)

metric_keys = ["has_docstring", "has_type_hints", "has_edge_case_handling", "has_return_statement", "syntactically_valid"]

for key in metric_keys:
    base_score = sum(r["metrics"][key] for r in base_results.values()) / len(PROMPTS) * 100
    v1_score = sum(r["metrics"][key] for r in v1_results.values()) / len(PROMPTS) * 100
    winner = "✅ V1" if v1_score > base_score else ("✅ BASE" if base_score > v1_score else "🟰 TIE")
    print(f"{key:35s} | Base: {base_score:5.1f}% | V1: {v1_score:5.1f}% | {winner}")

base_avg_loc = sum(r["metrics"]["lines_of_code"] for r in base_results.values()) / len(PROMPTS)
v1_avg_loc = sum(r["metrics"]["lines_of_code"] for r in v1_results.values()) / len(PROMPTS)
print(f"{'avg_lines_of_code':35s} | Base: {base_avg_loc:5.1f}  | V1: {v1_avg_loc:5.1f}")

base_avg_tps = sum(r["tokens_per_second"] for r in base_results.values()) / len(PROMPTS)
v1_avg_tps = sum(r["tokens_per_second"] for r in v1_results.values()) / len(PROMPTS)
print(f"{'avg_tokens_per_second':35s} | Base: {base_avg_tps:5.1f}  | V1: {v1_avg_tps:5.1f}")

AGGREGATE METRICS SUMMARY
has_docstring                       | Base:   0.0% | V1:  90.0% | ✅ V1
has_type_hints                      | Base:   0.0% | V1:  10.0% | ✅ V1
has_edge_case_handling              | Base:  60.0% | V1:  70.0% | ✅ V1
has_return_statement                | Base:  90.0% | V1:  90.0% | 🟰 TIE
syntactically_valid                 | Base:   0.0% | V1: 100.0% | ✅ V1
avg_lines_of_code                   | Base:  19.9  | V1:  10.2
avg_tokens_per_second               | Base:  29.1  | V1:   2.0


## 7. Save Results

The final cell writes the raw comparison output to `evaluation_results.json` so the results can be reviewed, shared, or used in a follow-up analysis report.


In [ ]:
import json

results = {
    "base_model": base_results,
    "v1_finetuned": v1_results,
}

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Results saved to evaluation_results.json")
print("Share evaluation_results.json for full analysis.")

Results saved to evaluation_results.json
Share evaluation_results.json for full analysis.


## 8. Evidence-first Analysis

The earlier cells create the raw experiment. The next cells turn `evaluation_results.json` into a cleaner evidence layer: behavior flags, richer tables, plots, and a practical interpretation of what fine-tuning actually changed.

This section can be rerun without loading either model, which makes it useful for reviewing results on machines with limited VRAM.

In [ ]:
import ast
import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

RESULT_CANDIDATES = [
    Path("Evaluation_results/evaluation_results.json"),
    Path("evaluation_results.json"),
]

for candidate in RESULT_CANDIDATES:
    if candidate.exists():
        RESULTS_PATH = candidate
        break
else:
    raise FileNotFoundError("Could not find evaluation_results.json")

with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    saved_results = json.load(f)

print(f"Loaded results from: {RESULTS_PATH}")
print(f"Prompts compared: {len(saved_results['base_model'])}")

## 9. Derived Quality Signals

The original metrics answer only part of the question. These derived signals look for the behavioral differences that matter in practice: whether the model returns raw code, whether prose or Markdown leaks into the answer, whether the code needs missing imports, whether the model generates class-style `self` context, and whether code extracted from Markdown fences would compile.

In [ ]:
def extract_code_candidate(output):
    fenced = re.findall(r"```(?:python)?\n(.*?)```", output, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        return "\n\n".join(block.strip() for block in fenced)
    return output.strip()


def syntax_valid(code):
    try:
        ast.parse(code)
        return True
    except SyntaxError:
        return False


def count_function_defs(code):
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return 0
    return sum(isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) for node in ast.walk(tree))


def issue_flags(output, code_candidate):
    lower = output.lower()
    return {
        "markdown_fence": "```" in output,
        "explanatory_prose": any(marker in lower for marker in ["certainly", "explanation", "example usage", "below is", "you can"]),
        "self_context": bool(re.search(r"\bself\b", code_candidate)),
        "missing_import_signal": any(token in code_candidate for token in ["math.", "re.", "reduce(", "List[", "Union["]) and "import " not in code_candidate,
        "truncated_or_cutoff": output.rstrip().endswith(("if", "for", "while", "from", "to", "is", ":", "-")),
    }


rows = []
for model_key, model_label in [("base_model", "Base"), ("v1_finetuned", "V1 fine-tuned")]:
    for prompt, record in saved_results[model_key].items():
        output = record["output"]
        code_candidate = extract_code_candidate(output)
        flags = issue_flags(output, code_candidate)
        metrics = record["metrics"]
        rows.append({
            "model": model_label,
            "prompt": prompt,
            "time_seconds": record["time_seconds"],
            "tokens_generated": record["tokens_generated"],
            "tokens_per_second": record["tokens_per_second"],
            "raw_syntax_valid": metrics["syntactically_valid"],
            "extracted_syntax_valid": syntax_valid(code_candidate),
            "has_docstring": metrics["has_docstring"],
            "has_type_hints": metrics["has_type_hints"],
            "has_edge_case_handling": metrics["has_edge_case_handling"],
            "has_return_statement": metrics["has_return_statement"],
            "lines_of_code": metrics["lines_of_code"],
            "candidate_lines": len([line for line in code_candidate.splitlines() if line.strip()]),
            "function_defs": count_function_defs(code_candidate),
            **flags,
        })

df = pd.DataFrame(rows)
bool_cols = [
    "raw_syntax_valid", "extracted_syntax_valid", "has_docstring", "has_type_hints",
    "has_edge_case_handling", "has_return_statement", "markdown_fence", "explanatory_prose",
    "self_context", "missing_import_signal", "truncated_or_cutoff",
]

display(df[[
    "model", "prompt", "raw_syntax_valid", "extracted_syntax_valid", "markdown_fence",
    "explanatory_prose", "self_context", "missing_import_signal", "candidate_lines",
    "tokens_per_second",
]])

## 10. Visual Comparison

These charts show the real shape of the difference. V1 is not simply "better" in every dimension: it is more code-shaped and syntactically usable, while the base model is faster and more conversational.

In [ ]:
quality_metrics = [
    "raw_syntax_valid",
    "extracted_syntax_valid",
    "has_docstring",
    "has_edge_case_handling",
    "has_return_statement",
]

quality_pct = df.groupby("model")[quality_metrics].mean().T * 100
ax = quality_pct.plot(kind="bar", figsize=(11, 5), color=["#4C78A8", "#F58518"])
ax.set_title("Code Quality Signals by Model")
ax.set_ylabel("Prompts with signal (%)")
ax.set_xlabel("")
ax.set_ylim(0, 110)
ax.legend(title="Model")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

issue_metrics = ["markdown_fence", "explanatory_prose", "self_context", "missing_import_signal", "truncated_or_cutoff"]
issue_pct = df.groupby("model")[issue_metrics].mean().T * 100
ax = issue_pct.plot(kind="bar", figsize=(11, 5), color=["#4C78A8", "#F58518"])
ax.set_title("Issue Signals by Model")
ax.set_ylabel("Prompts with issue signal (%)")
ax.set_xlabel("")
ax.set_ylim(0, 110)
ax.legend(title="Model")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby("model")["tokens_per_second"].mean().plot(kind="bar", ax=axes[0], color=["#4C78A8", "#F58518"])
axes[0].set_title("Average Generation Speed")
axes[0].set_ylabel("Tokens per second")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

df.groupby("model")["candidate_lines"].mean().plot(kind="bar", ax=axes[1], color=["#4C78A8", "#F58518"])
axes[1].set_title("Average Extracted Code Length")
axes[1].set_ylabel("Non-empty lines")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

display(quality_pct.round(1))
display(issue_pct.round(1))

## 11. Prompt-level Diagnosis

The prompt-level table is meant to guide improvement work. It separates wins caused by real fine-tuning behavior from wins caused by the base model's chat formatting being counted as invalid raw Python.

In [ ]:
score_cols = [
    "raw_syntax_valid",
    "extracted_syntax_valid",
    "has_docstring",
    "has_edge_case_handling",
    "has_return_statement",
]
penalty_cols = ["markdown_fence", "explanatory_prose", "self_context", "missing_import_signal", "truncated_or_cutoff"]

df["quality_score"] = df[score_cols].sum(axis=1) - df[penalty_cols].sum(axis=1)
pivot = df.pivot(index="prompt", columns="model", values="quality_score").reset_index()
pivot["delta_v1_minus_base"] = pivot["V1 fine-tuned"] - pivot["Base"]
pivot = pivot.sort_values("delta_v1_minus_base", ascending=False)

ax = pivot.plot(
    x="prompt",
    y="delta_v1_minus_base",
    kind="barh",
    figsize=(10, 6),
    color="#54A24B",
    legend=False,
)
ax.set_title("Prompt-level Quality Score Delta: V1 minus Base")
ax.set_xlabel("Positive means V1 was cleaner on the derived rubric")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

diagnosis = df.pivot(index="prompt", columns="model", values=[
    "raw_syntax_valid", "extracted_syntax_valid", "markdown_fence", "explanatory_prose",
    "self_context", "missing_import_signal", "candidate_lines", "tokens_per_second", "quality_score"
])
display(pivot)
display(diagnosis)

## 12. Final Interpretation and Improvement Intuition

### What changed after fine-tuning

V1 learned the response shape we wanted. It usually answers with direct Python code instead of a chat-style tutorial. That is why its raw syntax validity is much better in this evaluation: the base model often produces Markdown fences, examples, and prose, while V1 more often emits a standalone function.

V1 also became more compact. The generated answers are shorter and closer to the desired artifact: a function body rather than a full explanation. That is real value for a code-generation UI because the user can copy or inspect the function immediately.

### What did not improve enough

The fine-tune did not fully solve semantic correctness. Some V1 answers are syntactically valid but still imperfect: a few depend on missing imports such as `math`, `re`, or `reduce`; some include `self` even when the prompt asks for a standalone function; and some solve a nearby task rather than the exact task.

The base model is still much faster in the recorded run. So the fine-tune improved format discipline and raw-code usability, but it did not improve speed, and it still needs stronger correctness checks.

### Practical conclusion

Fine-tuning added real value, but mainly in output style alignment: V1 behaves more like a Python-function generator and less like a general chat assistant. The next improvement should target functional reliability, not just prettier or shorter code.

### Best next steps

1. Add executable unit tests for every evaluation prompt.
2. Add a metric for missing imports and unwanted `self` usage.
3. Train or filter examples toward standalone functions, not class methods unless requested.
4. Expand the evaluation set beyond ten prompts and include harder edge cases.
5. Compare three outputs: raw base, code-extracted base, and V1. That separates formatting gains from true coding gains.